# Fixed-Height Study - Notebook 2 of 3: Run Phyloformer

## ▶ RUNTIME: **GPU (T4)**
In Colab: **Runtime → Change runtime type → T4 GPU**. Setup stops with a clear message if no GPU is attached.

This notebook runs **Phyloformer only** on the trees and sequences saved by Notebook 1, and pushes the results. Only this notebook uses the GPU. Notebook 1 must have finished first.

**The three notebooks (run in this order)**

| # | Notebook | Runtime | What it does |
|---|---|---|---|
| 1 | `1_Generate_Trees_and_Sequences` | **CPU** | Simulates the trees and the sequences and saves them to the team repo |
| 2 | `2_Run_Phyloformer` | **GPU** | Runs Phyloformer on the saved data and pushes the results |
| 3 | `3_Run_Other_Methods` | **CPU** | Runs FastME, FastTree and IQ-TREE on the same saved data and pushes the results |

Notebook 1 must finish (and push) first. Notebooks 2 and 3 are independent of each other, so they can run at the same time in separate sessions.

## What this notebook does

For every grid cell (tree size x sequence length x model) it takes the saved alignments, runs Phyloformer, builds a tree from Phyloformer's distances with FastME, and compares that tree with the true tree.

**Timing (per tree, so it matches the paper's Phyloformer timing = Phyloformer + FastME)**
- `infer_seconds`: Phyloformer's forward pass only. Loading the model is not counted, and the GPU is synchronised before the clock is read.
- `fastme_seconds`: FastME building the tree from the distances.
- `runtime_seconds`: the two added together.

**GPU memory (recorded two independent ways, and the run stops if either reads zero)**
- `peak_alloc_mb`: the most memory the tensors needed while this alignment was processed (includes the model weights).
- `peak_reserved_mb`: the most memory PyTorch held. This is the number that decides whether a run fits on the card.
- `gpu_nvsmi_peak_mb`: peak read from `nvidia-smi` over the whole cell, minus the level before it started (a cross-check from outside the process; it also includes PyTorch's own start-up memory, so it is larger).

**Out of memory:** if one alignment is too big for the GPU, only that alignment fails. It is recorded in `failures_rate..._phyloformer.csv` and the rest of the cell carries on. Larger trees are the likely cases: on a T4, expect trouble from around 100 tips upwards, and this grid goes up to 250.


**What is in each results row (one row per tree)**

| Column | Meaning |
|---|---|
| `kf_score`, `rf`, `norm_rf`, `weighted_rf` | Kuhner-Felsenstein and Robinson-Foulds distances to the true tree |
| `mae` | Mean absolute error over all pairwise tip-to-tip distances (true vs inferred) |
| `runtime_seconds` | Time for **this one tree** (see each notebook for exactly what is timed) |
| `n_seqs`, `n_unique_sequences`, `n_duplicates` | Duplicate sequences in the alignment (`n_duplicates` = extra copies = `n_seqs - n_unique_sequences`) |
| `true_tree_height`, `true_total_length` | Size of the true tree (KF and MAE grow with it, so keep it for later normalising) |
| `seq_length_actual` | The alignment length that was really produced |

**Where things live in the team repo.** The code (setup files and notebooks) sits together in one folder. The data and the results go in their own folders, each with a `fixed_height_all_methods` subfolder, so earlier rounds are never touched.

```
fixed_height_all_methods/
├── setup/            the three helper files (setup_pipeline.py, fh_pipeline.py, pf_infer_timed.py)
└── notebooks/        these three notebooks

simulated_data/fixed_height_all_methods/
├── trees/true/clock_rate=R/tree_size=N/                          true trees         (notebook 1)
├── sequences/clock_rate=R/model=M/tree_size=N/seq_length=L/      alignments         (notebook 1)
└── trees/inferred/clock_rate=R/model=M/tree_size=N/seq_length=L/method=X/   trees built by each method (notebooks 2 and 3)

results/fixed_height_all_methods/
├── treesize_seqlength_rateR_{lg|cherry}_phyloformer_results.csv   (notebook 2)
├── treesize_seqlength_rateR_{lg|cherry}_baselines_results.csv     (notebook 3)
└── failures_rateR_{generate|phyloformer|baselines}.csv            anything that failed, and why
```

**If the session disconnects (or the GPU limit runs out):** start a new session and run from the top. Cells already in the results file are skipped. Results are pushed after every tree size.

## 1. Settings

In [ ]:
import os
BASE_DIR = globals().get("BASE_DIR", os.getcwd())   # folder that will hold Phyloformer/ and team_repo/ (stays fixed if cells are re-run)

# ---- Settings: change ONLY CLOCK_RATE (use your own rate; same value in all three notebooks) ----
CLOCK_RATE = 0.1          # Siqi (0.1, 5), Deep (0.5, 4), Mihir (1, 3.5), Swetha (1.5, 3), Darren (2, 2.5)
TREE_HEIGHT = 1.0         # every tree is rescaled to this height BEFORE the clock rate is applied
MODELS = ["LG", "Cherry"] # substitution models the sequences were simulated under
TREE_SIZES = [50, 100, 150, 200, 250]       # number of tips
SEQ_LENGTHS = [250, 500, 750, 1000, 1250]     # alignment length
N_TREES = 20              # replicate trees per tree size

## 2. Setup
Installs what is needed and checks that a GPU is attached and PyTorch can see it.

In [ ]:
# One-time setup. Prints a short checklist (details go to setup_log.txt).
# If it ends with "SETUP FAILED", fix the [FAIL] line and run this cell again.
import os
os.chdir(BASE_DIR)
!test -d team_repo || (git clone -q --filter=blob:none --sparse https://github.com/ACKarl/EPTI.git team_repo && git -C team_repo sparse-checkout set fixed_height_all_methods/setup)
!python team_repo/fixed_height_all_methods/setup/setup_pipeline.py --stage 2 --clock-rate {CLOCK_RATE}

In [ ]:
import os
import sys

PF_DIR = os.path.join(BASE_DIR, "Phyloformer")      # the tool being evaluated
TEAM_REPO = os.path.join(BASE_DIR, "team_repo")     # our shared repo (data and results are saved here)
sys.path.insert(0, os.path.join(TEAM_REPO, "fixed_height_all_methods", "setup"))
import fh_pipeline as fh                    # our helper functions (readable on GitHub: fixed_height_all_methods/setup/fh_pipeline.py)

WORK_DIR = os.path.join(PF_DIR, "data", "fh_work")   # scratch space for temporary files (wiped for every grid cell)

os.chdir(PF_DIR)                            # the repo's own scripts expect to run from this folder

## 3. GitHub login
Asked once, so the run is not interrupted later.

In [ ]:
# Asked ONCE per session. The token is kept in memory only (never written to a file or shown).
# It is used to push after every tree size, so a disconnect never costs more than one tree size.
creds = fh.ask_github_login(TEAM_REPO)

## 4. Sanity check (do this before the real run)

Runs Phyloformer on the repo's own 20 test alignments. It must give a mean KF distance of about **0.333** (the repo's published result) and **non-zero GPU numbers**. If it does not, stop and fix the environment first.

In [ ]:
fh.sanity_check_phyloformer(PF_DIR, WORK_DIR)

## 5. Run Phyloformer on the saved data

For each tree size, sequence length and model: skip the cell if it is already done, otherwise run Phyloformer, save the results, and log any failures. Pushes after every tree size.

In [ ]:
CHECKPOINTS = {"LG": "models/pf.ckpt", "Cherry": "models/pf_cherry.ckpt"}   # the model trained for each data type
RDIR = fh.results_dir(TEAM_REPO)

for size in TREE_SIZES:
    print(f"=== tree size {size} ===")
    for seqlen in SEQ_LENGTHS:
        for model in MODELS:
            params = {"model": model, "clock_rate": CLOCK_RATE, "tree_height": TREE_HEIGHT,
                      "tree_size": size, "seq_length": seqlen}
            fname = fh.results_filename(CLOCK_RATE, model, "phyloformer")
            cell = {"model": model, "tree_size": size, "seq_length": seqlen}

            # Resume: skip cells whose results are already saved.
            if fh.cell_done(RDIR, fname, params, ["Phyloformer"], N_TREES):
                print(f"  {model}, seq_length={seqlen}: already done, skipping")
                continue
            # The data comes from Notebook 1.
            if not fh.list_files(fh.sequences_dir(TEAM_REPO, CLOCK_RATE, model, size, seqlen), fh.ALN_EXTS):
                print(f"  {model}, seq_length={seqlen}: no saved sequences (run Notebook 1 first)")
                fh.save_failures(RDIR, CLOCK_RATE, "phyloformer", cell,
                                 [{"method": "Phyloformer", "tree_id": "ALL", "reason": "no saved sequences"}])
                continue

            print(f"  {model}, seq_length={seqlen}")
            try:
                df, fails = fh.run_phyloformer_cell(PF_DIR, TEAM_REPO, CLOCK_RATE, model, size, seqlen,
                                                    N_TREES, CHECKPOINTS[model], WORK_DIR)
            except fh.GpuNotRecordedError:
                raise        # GPU not being measured: stop everything, do not waste the remaining cells
            except Exception as e:
                df, fails = None, [{"method": "Phyloformer", "tree_id": "ALL", "reason": str(e)[:200]}]

            if df is not None:
                fh.save_results(df, params, fname, RDIR)
            fh.save_failures(RDIR, CLOCK_RATE, "phyloformer", cell, fails)

    try:
        fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                          f"Phyloformer results: clock rate {CLOCK_RATE}, tree size {size}", creds)
    except Exception as e:
        print(f"  !! PUSH FAILED (results are safe locally, the final cell will try again): {str(e)[:200]}")

print("Finished.")

## 6. Check the results are complete
Lists any grid cell with fewer than 20 trees, and summarises the logged failures.

In [ ]:
fh.completeness_report(TEAM_REPO, CLOCK_RATE, "phyloformer", MODELS, TREE_SIZES, SEQ_LENGTHS,
                       lambda model: ["Phyloformer"], N_TREES)

## 7. Final push
Pushes anything not yet in the repo (safe to run even if everything is already pushed).

In [ ]:
fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                  f"Phyloformer results: clock rate {CLOCK_RATE} (final)", creds)